# Data Profiling
## Indonesia Palm Oil Supply Chain Dataset

Notebook ini digunakan untuk melakukan profiling awal terhadap dataset
**Indonesia Palm Oil Supply Chain** dari Trase.

Profiling dilakukan sebagai bagian dari **Milestone 1 Tugas 1 Analisis Big Data 2026**.

### Tujuan

1. Memastikan dataset memenuhi ketentuan ukuran tugas.
2. Mengetahui struktur dan karakteristik dataset.
3. Mengidentifikasi nama kolom dan tipe data.
4. Memeriksa missing values secara awal.
5. Memeriksa data duplikat.
6. Mengetahui jumlah nilai unik pada setiap kolom.
7. Melakukan profiling sederhana menggunakan Polars.
8. Melakukan analisis SQL awal menggunakan DuckDB.

### Informasi Dataset

- **Nama:** Indonesia Palm Oil - Supply Chain Data
- **Sumber:** Trase
- **Periode:** 2013–2022
- **File:** `indonesia_palm_oil_v1_2_5.csv`
- **Format:** CSV
- **Unit analisis:** Data rantai pasok minyak kelapa sawit Indonesia

### Sumber Dataset

https://trase.earth/open-data/datasets/supply-chains-indonesia-palm-oil

In [1]:
import polars as pl
import duckdb
import os
import time

print("Polars version :", pl.__version__)
print("DuckDB version :", duckdb.__version__)

Polars version : 2.0.0
DuckDB version : 1.5.6


In [3]:
from pathlib import Path

DATA_PATH = Path("../data/raw/indonesia_palm_oil_v1_2_5.csv")

print("Dataset:")
print(DATA_PATH)

print("\nFile tersedia:", DATA_PATH.exists())

if DATA_PATH.exists():
    file_size = DATA_PATH.stat().st_size

    print(
        "Ukuran file:",
        f"{file_size / (1024**2):,.2f} MB"
    )
else:
    print("ERROR: File dataset tidak ditemukan.")

Dataset:
../data/raw/indonesia_palm_oil_v1_2_5.csv

File tersedia: True
Ukuran file: 721.83 MB


In [4]:
file_size = DATA_PATH.stat().st_size

file_size_mb = file_size / (1024 ** 2)
file_size_gb = file_size / (1024 ** 3)

print(f"Ukuran dataset : {file_size_mb:,.2f} MB")
print(f"Ukuran dataset : {file_size_gb:,.2f} GB")

Ukuran dataset : 721.83 MB
Ukuran dataset : 0.70 GB


In [5]:
start_time = time.time()

lf = pl.scan_csv(
    DATA_PATH,
    infer_schema_length=10_000,
    ignore_errors=False
)

elapsed = time.time() - start_time

print("LazyFrame berhasil dibuat.")
print(f"Waktu inisialisasi: {elapsed:.2f} detik")

LazyFrame berhasil dibuat.
Waktu inisialisasi: 0.02 detik


In [6]:
schema = lf.collect_schema()

schema

Schema([('year', Int64),
        ('country_of_production', String),
        ('product_type', String),
        ('forest_500_palm_oil', String),
        ('zero_deforestation_indonesia_palm_oil', String),
        ('province_of_production', String),
        ('kabupaten_of_production', String),
        ('mill', String),
        ('mill_trase_id', String),
        ('mill_group', String),
        ('refinery', String),
        ('refinery_trase_id', String),
        ('refinery_group', String),
        ('exporter', String),
        ('exporter_group', String),
        ('port_of_export', String),
        ('importer', String),
        ('importer_group', String),
        ('country_of_first_import', String),
        ('economic_bloc', String),
        ('province_of_production_trase_id', String),
        ('kabupaten_of_production_trase_id', String),
        ('country_of_production_trase_id', String),
        ('country_of_first_import_trase_id', String),
        ('palm_oil_deforestation_10_year_total_exp

In [7]:
schema_df = pl.DataFrame({
    "column": list(schema.names()),
    "dtype": [
        str(schema[name])
        for name in schema.names()
    ]
})

schema_df

column,dtype
str,str
"""year""","""Int64"""
"""country_of_production""","""String"""
"""product_type""","""String"""
"""forest_500_palm_oil""","""String"""
"""zero_deforestation_indonesia_p…","""String"""
…,…
"""gross_emission_luc_exposure""","""String"""
"""emissions_fire_on_peat_exposur…","""String"""
"""volume""","""Float64"""


In [8]:
column_count = len(schema.names())

print("Jumlah kolom:", column_count)

Jumlah kolom: 34


In [9]:
start_time = time.time()

row_count = (
    lf.select(
        pl.len().alias("jumlah_baris")
    )
    .collect()
    .item()
)

elapsed = time.time() - start_time

print("Jumlah baris:", f"{row_count:,}")
print(f"Waktu proses: {elapsed:.2f} detik")

Jumlah baris: 1,694,907
Waktu proses: 6.95 detik


In [10]:
print("HASIL VALIDASI DATASET")
print("=" * 40)

print(f"Jumlah baris : {row_count:,}")
print(f"Ukuran file : {file_size_mb:,.2f} MB")

if row_count > 1_000_000:
    print("\n✓ MEMENUHI: jumlah baris > 1.000.000")
elif file_size_mb >= 500:
    print("\n✓ MEMENUHI: ukuran file >= 500 MB")
else:
    print("\n✗ BELUM MEMENUHI ketentuan ukuran dataset")

HASIL VALIDASI DATASET
Jumlah baris : 1,694,907
Ukuran file : 721.83 MB

✓ MEMENUHI: jumlah baris > 1.000.000


In [11]:
sample = (
    lf
    .head(10)
    .collect()
)

sample

year,country_of_production,product_type,forest_500_palm_oil,zero_deforestation_indonesia_palm_oil,province_of_production,kabupaten_of_production,mill,mill_trase_id,mill_group,refinery,refinery_trase_id,refinery_group,exporter,exporter_group,port_of_export,importer,importer_group,country_of_first_import,economic_bloc,province_of_production_trase_id,kabupaten_of_production_trase_id,country_of_production_trase_id,country_of_first_import_trase_id,palm_oil_deforestation_10_year_total_exposure,palm_oil_deforestation_annual_exposure,total_emission_exposure,net_emission_luc_exposure,emission_subsidence_exposure,gross_emission_luc_exposure,emissions_fire_on_peat_exposure,volume,oil_palm_ha,fob
i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,str,str
2013,"""INDONESIA""","""PALM OIL""","""NOT ASSESSED""","""UNKNOWN""","""UNKNOWN""","""UNKNOWN""","""UNKNOWN""","""ID-PALM-MILL-X""","""UNKNOWN""","""UNKNOWN""","""ID-PALM-REFINERY-X""","""UNKNOWN""","""ACCESS BALI""","""ACCESS BALI""","""TANJUNG PRIOK""","""JOV-HENTON LTD""","""JOV-HENTON LTD""","""NIGERIA""","""NIGERIA""","""ID-XX""","""ID-XXXX""","""ID""","""NG""",null,null,null,null,null,null,null,22.8,null,null
2013,"""INDONESIA""","""PALM OIL""","""NOT ASSESSED""","""UNKNOWN""","""UNKNOWN""","""UNKNOWN""","""UNKNOWN""","""ID-PALM-MILL-X""","""UNKNOWN""","""UNKNOWN""","""ID-PALM-REFINERY-X""","""UNKNOWN""","""ADEI PLANTATION & INDUSTRY""","""KUALA LUMPUR KEPONG (KLK)""","""DUMAI""","""ACALPO""","""ACALPO""","""INDIA""","""INDIA""","""ID-XX""","""ID-XXXX""","""ID""","""IN""",null,null,null,null,null,null,null,3000.0,null,null
2013,"""INDONESIA""","""PALM OIL""","""NOT ASSESSED""","""UNKNOWN""","""UNKNOWN""","""UNKNOWN""","""UNKNOWN""","""ID-PALM-MILL-X""","""UNKNOWN""","""UNKNOWN""","""ID-PALM-REFINERY-X""","""UNKNOWN""","""ADEI PLANTATION & INDUSTRY""","""KUALA LUMPUR KEPONG (KLK)""","""DUMAI""","""BUNGE""","""BUNGE""","""INDIA""","""INDIA""","""ID-XX""","""ID-XXXX""","""ID""","""IN""",null,null,null,null,null,null,null,1000.0,null,null
2013,"""INDONESIA""","""PALM OIL""","""NOT ASSESSED""","""UNKNOWN""","""UNKNOWN""","""UNKNOWN""","""UNKNOWN""","""ID-PALM-MILL-X""","""UNKNOWN""","""UNKNOWN""","""ID-PALM-REFINERY-X""","""UNKNOWN""","""ADEI PLANTATION & INDUSTRY""","""KUALA LUMPUR KEPONG (KLK)""","""DUMAI""","""CARGILL""","""CARGILL""","""INDIA""","""INDIA""","""ID-XX""","""ID-XXXX""","""ID""","""IN""",null,null,null,null,null,null,null,1000.0,null,null
2013,"""INDONESIA""","""PALM OIL""","""NOT ASSESSED""","""UNKNOWN""","""UNKNOWN""","""UNKNOWN""","""UNKNOWN""","""ID-PALM-MILL-X""","""UNKNOWN""","""UNKNOWN""","""ID-PALM-REFINERY-X""","""UNKNOWN""","""ADEI PLANTATION & INDUSTRY""","""KUALA LUMPUR KEPONG (KLK)""","""DUMAI""","""CARGILL""","""CARGILL""","""TANZANIA""","""TANZANIA""","""ID-XX""","""ID-XXXX""","""ID""","""TZ""",null,null,null,null,null,null,null,500.0,null,null
2013,"""INDONESIA""","""PALM OIL""","""NOT ASSESSED""","""UNKNOWN""","""UNKNOWN""","""UNKNOWN""","""UNKNOWN""","""ID-PALM-MILL-X""","""UNKNOWN""","""UNKNOWN""","""ID-PALM-REFINERY-X""","""UNKNOWN""","""ADEI PLANTATION & INDUSTRY""","""KUALA LUMPUR KEPONG (KLK)""","""DUMAI""","""JOSOVINA COMMODITIES""","""JOSOVINA COMMODITIES""","""GHANA""","""GHANA""","""ID-XX""","""ID-XXXX""","""ID""","""GH""",null,null,null,null,null,null,null,2500.0,null,null
2013,"""INDONESIA""","""PALM OIL""","""NOT ASSESSED""","""UNKNOWN""","""UNKNOWN""","""UNKNOWN""","""UNKNOWN""","""ID-PALM-MILL-X""","""UNKNOWN""","""UNKNOWN""","""ID-PALM-REFINERY-X""","""UNKNOWN""","""ADEI PLANTATION & INDUSTRY""","""KUALA LUMPUR KEPONG (KLK)""","""DUMAI""","""JOSOVINA COMMODITIES""","""JOSOVINA COMMODITIES""","""NETHERLANDS""","""EUROPEAN UNION""","""ID-XX""","""ID-XXXX""","""ID""","""NL""",null,null,null,null,null,null,null,500.0,null,null
2013,"""INDONESIA""","""PALM OIL""","""NOT ASSESSED""","""UNKNOWN""","""UNKNOWN""","""UNKNOWN""","""UNKNOWN""","""ID-PALM-MILL-X""","""UNKNOWN

In [12]:
schema_df

column,dtype
str,str
"""year""","""Int64"""
"""country_of_production""","""String"""
"""product_type""","""String"""
"""forest_500_palm_oil""","""String"""
"""zero_deforestation_indonesia_p…","""String"""
…,…
"""gross_emission_luc_exposure""","""String"""
"""emissions_fire_on_peat_exposur…","""String"""
"""volume""","""Float64"""


In [13]:
missing_profile = (
    lf
    .select([
        pl.col(column)
        .null_count()
        .alias(column)
        for column in schema.names()
    ])
    .collect()
)

missing_profile

year,country_of_production,product_type,forest_500_palm_oil,zero_deforestation_indonesia_palm_oil,province_of_production,kabupaten_of_production,mill,mill_trase_id,mill_group,refinery,refinery_trase_id,refinery_group,exporter,exporter_group,port_of_export,importer,importer_group,country_of_first_import,economic_bloc,province_of_production_trase_id,kabupaten_of_production_trase_id,country_of_production_trase_id,country_of_first_import_trase_id,palm_oil_deforestation_10_year_total_exposure,palm_oil_deforestation_annual_exposure,total_emission_exposure,net_emission_luc_exposure,emission_subsidence_exposure,gross_emission_luc_exposure,emissions_fire_on_peat_exposure,volume,oil_palm_ha,fob
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,10233,10233,1025290,1025290,1025290,1025290,1025290,0,10233,10957


In [14]:
missing_summary = pl.DataFrame({
    "column": schema.names(),
    "missing_count": [
        missing_profile[column][0]
        for column in schema.names()
    ]
})

missing_summary = missing_summary.with_columns(
    (
        pl.col("missing_count") / row_count * 100
    ).alias("missing_percentage")
)

missing_summary.sort(
    "missing_count",
    descending=True
)

column,missing_count,missing_percentage
str,i64,f64
"""total_emission_exposure""",1025290,60.492405
"""net_emission_luc_exposure""",1025290,60.492405
"""emission_subsidence_exposure""",1025290,60.492405
"""gross_emission_luc_exposure""",1025290,60.492405
"""emissions_fire_on_peat_exposur…",1025290,60.492405
…,…,…
"""province_of_production_trase_i…",0,0.0
"""kabupaten_of_production_trase_…",0,0.0
"""country_of_production_trase_id""",0,0.0


In [15]:
missing_columns = (
    missing_summary
    .filter(
        pl.col("missing_count") > 0
    )
    .sort(
        "missing_count",
        descending=True
    )
)

print(
    "Jumlah kolom dengan missing value:",
    missing_columns.height
)

missing_columns

Jumlah kolom dengan missing value: 9


column,missing_count,missing_percentage
str,i64,f64
"""total_emission_exposure""",1025290,60.492405
"""net_emission_luc_exposure""",1025290,60.492405
"""emission_subsidence_exposure""",1025290,60.492405
"""gross_emission_luc_exposure""",1025290,60.492405
"""emissions_fire_on_peat_exposur…",1025290,60.492405
"""fob""",10957,0.646466
"""palm_oil_deforestation_10_year…",10233,0.60375
"""palm_oil_deforestation_annual_…",10233,0.60375
"""oil_palm_ha""",10233,0.60375


In [16]:
start_time = time.time()

unique_row_count = (
    lf
    .unique()
    .select(
        pl.len().alias("unique_rows")
    )
    .collect()
    .item()
)

duplicate_count = row_count - unique_row_count

elapsed = time.time() - start_time

print("Jumlah baris       :", f"{row_count:,}")
print("Jumlah baris unik  :", f"{unique_row_count:,}")
print("Perkiraan duplikat :", f"{duplicate_count:,}")
print(f"Waktu proses       : {elapsed:.2f} detik")

Jumlah baris       : 1,694,907
Jumlah baris unik  : 1,694,907
Perkiraan duplikat : 0
Waktu proses       : 38.91 detik


In [17]:
unique_counts = (
    lf
    .select([
        pl.col(column)
        .n_unique()
        .alias(column)
        for column in schema.names()
    ])
    .collect()
)

unique_summary = pl.DataFrame({
    "column": schema.names(),
    "unique_values": [
        unique_counts[column][0]
        for column in schema.names()
    ]
})

unique_summary.sort(
    "unique_values",
    descending=True
)

column,unique_values
str,i64
"""fob""",1569223
"""oil_palm_ha""",1482977
"""volume""",1467542
"""palm_oil_deforestation_10_year…",693684
"""palm_oil_deforestation_annual_…",320648
…,…
"""forest_500_palm_oil""",7
"""zero_deforestation_indonesia_p…",4
"""product_type""",2


In [18]:
numeric_columns = [
    name
    for name, dtype in schema.items()
    if dtype.is_numeric()
]

print("Kolom numerik:")
for column in numeric_columns:
    print("-", column)

print("\nJumlah kolom numerik:", len(numeric_columns))

Kolom numerik:
- year
- volume

Jumlah kolom numerik: 2


In [19]:
categorical_columns = [
    name
    for name, dtype in schema.items()
    if dtype == pl.String
]

print("Kolom kategorikal/string:")

for column in categorical_columns:
    print("-", column)

print(
    "\nJumlah kolom kategorikal:",
    len(categorical_columns)
)

Kolom kategorikal/string:
- country_of_production
- product_type
- forest_500_palm_oil
- zero_deforestation_indonesia_palm_oil
- province_of_production
- kabupaten_of_production
- mill
- mill_trase_id
- mill_group
- refinery
- refinery_trase_id
- refinery_group
- exporter
- exporter_group
- port_of_export
- importer
- importer_group
- country_of_first_import
- economic_bloc
- province_of_production_trase_id
- kabupaten_of_production_trase_id
- country_of_production_trase_id
- country_of_first_import_trase_id
- palm_oil_deforestation_10_year_total_exposure
- palm_oil_deforestation_annual_exposure
- total_emission_exposure
- net_emission_luc_exposure
- emission_subsidence_exposure
- gross_emission_luc_exposure
- emissions_fire_on_peat_exposure
- oil_palm_ha
- fob

Jumlah kolom kategorikal: 32


In [20]:
if numeric_columns:

    numeric_summary = (
        lf
        .select(numeric_columns)
        .collect()
        .describe()
    )

    numeric_summary

else:
    print("Tidak terdapat kolom numerik.")

In [21]:
if "year" in schema.names():

    year_profile = (
        lf
        .group_by("year")
        .agg(
            pl.len().alias("jumlah_baris")
        )
        .sort("year")
        .collect()
    )

    year_profile

else:
    print("Kolom 'year' tidak ditemukan.")

In [22]:
if "year" in schema.names():

    year_range = (
        lf
        .select([
            pl.col("year").min().alias("tahun_minimum"),
            pl.col("year").max().alias("tahun_maksimum")
        ])
        .collect()
    )

    year_range

In [23]:
if "country_of_production" in schema.names():

    country_profile = (
        lf
        .group_by("country_of_production")
        .agg(
            pl.len().alias("jumlah_baris")
        )
        .sort(
            "jumlah_baris",
            descending=True
        )
        .head(20)
        .collect()
    )

    country_profile

else:
    print(
        "Kolom 'country_of_production' tidak ditemukan."
    )

In [24]:
if "product_type" in schema.names():

    product_profile = (
        lf
        .group_by("product_type")
        .agg(
            pl.len().alias("jumlah_baris")
        )
        .sort(
            "jumlah_baris",
            descending=True
        )
        .head(20)
        .collect()
    )

    product_profile

else:
    print(
        "Kolom 'product_type' tidak ditemukan."
    )

In [25]:
con = duckdb.connect()

print("DuckDB berhasil terhubung.")

DuckDB berhasil terhubung.


In [26]:
duckdb_result = con.execute(f"""
    SELECT *
    FROM read_csv_auto('{DATA_PATH}')
    LIMIT 10
""").df()

duckdb_result

,year,country_of_production,product_type,forest_500_palm_oil,zero_deforestation_indonesia_palm_oil,province_of_production,kabupaten_of_production,mill,mill_trase_id,mill_group,...,palm_oil_deforestation_10_year_total_exposure,palm_oil_deforestation_annual_exposure,total_emission_exposure,net_emission_luc_exposure,emission_subsidence_exposure,gross_emission_luc_exposure,emissions_fire_on_peat_exposure,volume,oil_palm_ha,fob
0,2013,INDONESIA,PALM OIL,NOT ASSESSED,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,ID-PALM-MILL-X,UNKNOWN,...,NaN,NaN,None,None,None,None,None,22.8,NaN,NaN
1,2013,INDONESIA,PALM OIL,NOT ASSESSED,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,ID-PALM-MILL-X,UNKNOWN,...,NaN,NaN,None,None,None,None,None,3000.0,NaN,NaN
2,2013,INDONESIA,PALM OIL,NOT ASSESSED,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,ID-PALM-MILL-X,UNKNOWN,...,NaN,NaN,None,None,None,None,None,1000.0,NaN,NaN
3,2013,INDONESIA,PALM OIL,NOT ASSESSED,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,ID-PALM-MILL-X,UNKNOWN,...,NaN,NaN,None,None,None,None,None,1000.0,NaN,NaN
4,2013,INDONESIA,PALM OIL,NOT ASSESSED,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,ID-PALM-MILL-X,UNKNOWN,...,NaN,NaN,None,None,None,None,None,500.0,NaN,NaN
5,2013,INDONESIA,PALM OIL,NOT ASSESSED,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,ID-PALM-MILL-X,UNKNOWN,...,NaN,NaN,None,None,None,None,None,2500.0,NaN,NaN
6,2013,INDONESIA,PALM OIL,NOT ASSESSED,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,ID-PALM-MILL-X,UNKNOWN,...,NaN,NaN,None,None,None,None,None,500.0,NaN,NaN
7,2013,INDONESIA,PALM OIL,NOT ASSESSED,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,ID-PALM-MILL-X,UNKNOWN,...,NaN,NaN,None,None,None,None,None,1000.0,NaN,NaN
8,2013,INDONESIA,PALM OIL,NOT ASSESSED,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,ID-PALM-MILL-X,UNKNOWN,...,NaN,NaN,None,None,None,None,None,500.0,NaN,NaN
9,2013,INDONESIA,PALM OIL,NOT ASSESSED,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,ID-PALM-MILL-X,UNKNOWN,...,NaN,NaN,None,None,None,None,None,3000.0,NaN,NaN


In [27]:
duckdb_count = con.execute(f"""
    SELECT COUNT(*) AS jumlah_baris
    FROM read_csv_auto('{DATA_PATH}')
""").df()

duckdb_count

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,jumlah_baris
0,1694907


In [28]:
summarize_result = con.execute(f"""
    SUMMARIZE
    SELECT *
    FROM read_csv_auto('{DATA_PATH}')
""").df()

summarize_result

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,year,BIGINT,2013,2022,11,2020.0519167128343,1.3540499690520635,2019,2020,2021,1694907,0.00
1,country_of_production,VARCHAR,INDONESIA,INDONESIA,1,NaN,NaN,NaN,NaN,NaN,1694907,0.00
2,product_type,VARCHAR,PALM OIL,REFINED PALM OIL,2,NaN,NaN,NaN,NaN,NaN,1694907,0.00
3,forest_500_palm_oil,VARCHAR,0,NOT ASSESSED,7,NaN,NaN,NaN,NaN,NaN,1694907,0.00
4,zero_deforestation_indonesia_palm_oil,VARCHAR,COMPANY COMMITMENT,UNKNOWN,4,NaN,NaN,NaN,NaN,NaN,1694907,0.00
5,province_of_production,VARCHAR,ACEH,UNKNOWN,32,NaN,NaN,NaN,NaN,NaN,1694907,0.00
6,kabupaten_of_production,VARCHAR,ACEH BARAT,WAY KANAN,282,NaN,NaN,NaN,NaN,NaN,1694907,0.00
7,mill,VARCHAR,ABDI BORNEO PLANTATIONS (ABDI),YUNI BERSAUDARA SEJAHTERA,1428,NaN,NaN,NaN,NaN,NaN,1694907,0.00
8,mill_trase_id,VARCHAR,ID-PALM-MILL-00001,ID-PALM-MILL-X,1287,NaN,NaN,NaN,NaN,NaN,1694907,0.00
9,mill_group,VARCHAR,ADIMULIA,ZONERGY,225,NaN,NaN,NaN,NaN,NaN,1694907,0.00


In [29]:
if "year" in schema.names():

    duckdb_year = con.execute(f"""
        SELECT
            year,
            COUNT(*) AS jumlah_data
        FROM read_csv_auto('{DATA_PATH}')
        GROUP BY year
        ORDER BY year
    """).df()

    duckdb_year

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## Perbandingan Polars dan DuckDB

Pada tahap profiling ini digunakan dua teknologi:

- **Polars** digunakan untuk membaca dan melakukan transformasi awal
  terhadap dataset dengan pendekatan `LazyFrame` melalui `scan_csv()`.
- **DuckDB** digunakan untuk melakukan analisis berbasis SQL terhadap
  dataset, termasuk perhitungan jumlah baris dan profiling menggunakan
  `SUMMARIZE`.

Pendekatan ini memungkinkan dataset berukuran besar dianalisis tanpa
bergantung pada Pandas sebagai alat utama manipulasi data.

In [31]:
total_missing = int(
    missing_summary["missing_count"].sum()
)

profiling_summary = pl.DataFrame({
    "Informasi": [
        "Nama Dataset",
        "Jumlah Baris",
        "Jumlah Kolom",
        "Ukuran File (MB)",
        "Jumlah Missing Value",
        "Jumlah Baris Duplikat",
        "Kolom Numerik",
        "Kolom Kategorikal"
    ],
    "Nilai": [
        "Indonesia Palm Oil - Supply Chain Data",
        f"{row_count:,}",
        f"{column_count:,}",
        f"{file_size_mb:,.2f}",
        f"{total_missing:,}",
        f"{duplicate_count:,}",
        f"{len(numeric_columns):,}",
        f"{len(categorical_columns):,}"
    ]
})

profiling_summary

Informasi,Nilai
str,str
"""Nama Dataset""","""Indonesia Palm Oil - Supply Ch…"
"""Jumlah Baris""","""1,694,907"""
"""Jumlah Kolom""","""34"""
"""Ukuran File (MB)""","""721.83"""
"""Jumlah Missing Value""","""5,168,106"""
"""Jumlah Baris Duplikat""","""0"""
"""Kolom Numerik""","""2"""
"""Kolom Kategorikal""","""32"""


# Kesimpulan

Berdasarkan profiling awal, dataset **Indonesia Palm Oil - Supply Chain
Data** merupakan dataset berukuran besar yang digunakan untuk
menganalisis informasi rantai pasok minyak kelapa sawit Indonesia.

Profiling dilakukan menggunakan **Polars** dan **DuckDB**. Polars
digunakan dengan pendekatan lazy evaluation melalui `scan_csv()` untuk
membaca dan melakukan profiling terhadap dataset, sedangkan DuckDB
digunakan untuk melakukan analisis berbasis SQL.

Beberapa aspek yang diperiksa meliputi ukuran dataset, jumlah baris dan
kolom, struktur data, tipe data, missing values, duplikasi, nilai unik,
statistik deskriptif, serta distribusi data berdasarkan tahun.

Hasil profiling ini menjadi dasar untuk tahap berikutnya. Proses
cleaning, validasi data yang lebih mendalam, penanganan missing value,
duplikasi, dan outlier akan dilakukan pada **Milestone 2** melalui
notebook `02_data_cleaning.ipynb`.

Dataset asli tetap disimpan pada folder `data/raw/` dan tidak diubah.